<h2 align="center">Lab Session 7. Large Language Models: Prompting and In-context Learning</h2>
<h3 style="display:block; margin-top:5px;" align="center">Natural Language and Information Retrieval</h3>
<h3 style="display:block; margin-top:5px;" align="center">Degree in Data Science</h3>
<h3 style="display:block; margin-top:5px;" align="center">ETSInf. Universitat Politècnica de València</h3>
<br>

### Put your names here

- Alejandro de las Nieves Duro

# DO THE WORK !!

In [7]:
!pip install -q transformers accelerate sentencepiece scikit-learn pandas tqdm

In [8]:
import os
import re
import ast
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_PATH = "/content/drive/MyDrive/LNR/P7/"
DATASET_PATH = "/content/drive/MyDrive/LNR/P7/EXIST2026_training.json"

OUTPUT_DIR = "/content/drive/MyDrive/LNR/P7/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cpu'

In [9]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
with open(DATASET_PATH, encoding="utf-8") as f:
    raw_exist = json.load(f)


LABELS_23 = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE"
]


def majority_binary(votes, positive_label, negative_label):
    counts = Counter(votes)
    if counts[positive_label] > counts[negative_label]:
        return positive_label
    if counts[negative_label] > counts[positive_label]:
        return negative_label
    return None


def majority_task22(votes):
    valid_votes = [vote for vote in votes if vote in {"DIRECT", "JUDGEMENTAL"}]
    counts = Counter(valid_votes)
    if counts["DIRECT"] > counts["JUDGEMENTAL"]:
        return "DIRECT"
    if counts["JUDGEMENTAL"] > counts["DIRECT"]:
        return "JUDGEMENTAL"
    return None


def union_task23(annotation_groups):
    labels = set()
    for group in annotation_groups:
        group_labels = group if isinstance(group, list) else [group]
        for label in group_labels:
            if label not in {"-", "UNKNOWN"}:
                labels.add(label)
    return sorted(labels)


rows_21 = []
rows_22 = []
rows_23 = []

for fallback_id, sample in raw_exist.items():
    sample_id = str(sample.get("id_EXIST", fallback_id))
    lang = sample["lang"]
    text = sample["text"]

    label_21 = majority_binary(sample["labels_task2_1"], "YES", "NO")

    if label_21 is None:
        continue

    rows_21.append({
        "id": sample_id,
        "lang": lang,
        "text": text,
        "label": label_21
    })

    if label_21 == "YES":
        label_22 = majority_task22(sample["labels_task2_2"])
        if label_22 is not None:
            rows_22.append({
                "id": sample_id,
                "lang": lang,
                "text": text,
                "label": label_22
            })

        labels_23 = union_task23(sample["labels_task2_3"])
        if labels_23:
            rows_23.append({
                "id": sample_id,
                "lang": lang,
                "text": text,
                "label": labels_23
            })


task21_df = pd.DataFrame(rows_21)
task22_df = pd.DataFrame(rows_22)
task23_df = pd.DataFrame(rows_23)


def split_english_dev(df, label_col="label", test_size=0.2):
    df_en = df[df["lang"] == "en"].reset_index(drop=True)

    if label_col == "label" and not isinstance(df_en[label_col].iloc[0], list):
        train_df, dev_df = train_test_split(
            df_en,
            test_size=test_size,
            random_state=SEED,
            stratify=df_en[label_col]
        )
    else:
        train_df, dev_df = train_test_split(
            df_en,
            test_size=test_size,
            random_state=SEED,
            shuffle=True
        )

    return train_df.reset_index(drop=True), dev_df.reset_index(drop=True)


train_21, dev_21 = split_english_dev(task21_df)
train_22, dev_22 = split_english_dev(task22_df)
train_23, dev_23 = split_english_dev(task23_df)

summary_df = pd.DataFrame([
    {"task": "2.1", "train": len(train_21), "dev": len(dev_21), "labels": train_21["label"].value_counts().to_dict()},
    {"task": "2.2", "train": len(train_22), "dev": len(dev_22), "labels": train_22["label"].value_counts().to_dict()},
    {"task": "2.3", "train": len(train_23), "dev": len(dev_23), "labels": "multi-label"},
])

display(summary_df)

,task,train,dev,labels
0,2.1,1363,341,"{'YES': 770, 'NO': 593}"
1,2.2,640,161,"{'DIRECT': 459, 'JUDGEMENTAL': 181}"
2,2.3,770,193,multi-label


In [12]:
MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print("Model loaded:", MODEL_NAME)
print("Device:", DEVICE)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/flan-t5-small
Device: cpu


In [13]:
def perform_incontext_classification(model, tokenizer, prompt, ntokens=16, nseq=1, temp=0.3):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=ntokens,
            num_return_sequences=nseq,
            do_sample=False,
            temperature=temp
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.strip()


def create_incontext_zero_prompt(task_description, query, context=None, role=None, output=None):
    prompt = ""

    if role is not None:
        prompt += f"You are an expert {role}.\n\n"

    prompt += f"Task: {task_description}\n\n"

    if context is not None:
        prompt += f"Context: {context}\n\n"

    if output is not None:
        prompt += f"Output Format: {output}\n\n"

    prompt += f"Input: {query}\n"
    prompt += "Output:"

    return prompt


def create_incontext_few_prompt(task_description, exemplars, query, context=None, role=None, output=None):
    prompt = ""

    if role is not None:
        prompt += f"You are an expert {role}.\n\n"

    prompt += f"Task: {task_description}\n\n"

    if context is not None:
        prompt += f"Context: {context}\n\n"

    prompt += "Examples:\n\n"

    for exemplar in exemplars:
        prompt += f"Input: {exemplar['input']}\n"
        prompt += f"Output: {exemplar['output']}\n\n"

    if output is not None:
        prompt += f"Output Format: {output}\n\n"

    prompt += f"Input: {query}\n"
    prompt += "Output:"

    return prompt


def stratified_sample(df, label_column, sample_size=2):
    sampled_parts = []

    for label, group in df.groupby(label_column):
        sampled_parts.append(
            group.sample(
                n=min(sample_size, len(group)),
                random_state=SEED
            )
        )

    return pd.concat(sampled_parts, ignore_index=True)

In [14]:
def postprocess_task21(output):
    clean = output.strip().upper()

    if clean.startswith("YES") or "YES" in clean.split():
        return "YES"
    if clean.startswith("NO") or "NO" in clean.split():
        return "NO"

    return "NO"


def postprocess_task22(output):
    clean = output.strip().upper()

    if clean.startswith("DIRECT") or "DIRECT" in clean:
        return "DIRECT"
    if clean.startswith("JUDGEMENTAL") or "JUDGEMENTAL" in clean:
        return "JUDGEMENTAL"

    return "DIRECT"


def postprocess_task23(output):
    clean = output.upper()
    predicted = []

    for label in LABELS_23:
        if label in clean:
            predicted.append(label)

    if not predicted:
        predicted = ["STEREOTYPING-DOMINANCE"]

    return sorted(list(set(predicted)))

In [15]:
def run_zero_shot_task21(model, tokenizer, dev_df, max_examples=None):
    role = "in social psychology and linguistics, specialized in detecting sexist language in social media"

    task = """
Decide whether the following text is sexist or not.
A text is sexist if it expresses sexist ideas, describes discrimination against women,
or criticizes a sexist situation.
Return YES if the text is sexist. Return NO if the text is not sexist.
"""

    output = "Return only one label: YES or NO."

    if max_examples is not None:
        dev_df = dev_df.head(max_examples).copy()

    predictions = []
    raw_outputs = []

    for _, row in tqdm(dev_df.iterrows(), total=len(dev_df)):
        prompt = create_incontext_zero_prompt(
            task_description=task,
            query=row["text"],
            role=role,
            output=output
        )

        response = perform_incontext_classification(
            model,
            tokenizer,
            prompt,
            ntokens=8,
            temp=0.2
        )

        pred = postprocess_task21(response)

        predictions.append(pred)
        raw_outputs.append(response)

    result_df = dev_df.copy()
    result_df["prediction"] = predictions
    result_df["raw_output"] = raw_outputs
    result_df["correct"] = result_df["label"] == result_df["prediction"]

    f1_positive = f1_score(
        result_df["label"],
        result_df["prediction"],
        pos_label="YES",
        average="binary"
    )

    print("Subtask 2.1 Zero-shot F1 positive class:", f1_positive)
    print(classification_report(result_df["label"], result_df["prediction"], digits=4))

    result_df.to_csv(os.path.join(OUTPUT_DIR, "task21_zero_shot_predictions.csv"), index=False)

    return result_df, f1_positive


task21_zero_df, task21_zero_f1 = run_zero_shot_task21(
    model,
    tokenizer,
    dev_21,
    max_examples=50
)

display(task21_zero_df.head())

  0%|          | 0/50 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Subtask 2.1 Zero-shot F1 positive class: 0.3
              precision    recall  f1-score   support

          NO     0.4103    0.7619    0.5333        21
         YES     0.5455    0.2069    0.3000        29

    accuracy                         0.4400        50
   macro avg     0.4779    0.4844    0.4167        50
weighted avg     0.4887    0.4400    0.3980        50



,id,lang,text,label,prediction,raw_output,correct
0,210610,en,I COMPLETELY SUBSCRIBE TO GENDER ROLES AND FIN...,YES,NO,No,False
1,211475,en,GOT PROMOTED AFTER 8 YEARS AS A FORKLIFT OPERA...,NO,NO,No,True
2,211142,en,When you need a man for a couple days!!! MAN |...,NO,NO,No,True
3,210200,en,THERE MUST BE SOME WAY TO BLAME BLACK MEN... D...,YES,NO,No,False
4,210262,en,EVERY MAN WHEN HE SEES NICE BOOBS imgflip.com,YES,NO,No,False


In [16]:
def build_fewshot_examples_single_label(train_df, examples_per_label=2):
    sampled = stratified_sample(train_df, "label", sample_size=examples_per_label)

    examples = []

    for _, row in sampled.iterrows():
        examples.append({
            "input": row["text"],
            "output": row["label"]
        })

    return examples


def run_few_shot_task22(model, tokenizer, train_df, dev_df, max_examples=None):
    role = "in social psychology and linguistics, specialized in the analysis of sexist messages"

    task = """
Classify the intention of a sexist text.
Use DIRECT if the text is sexist by itself or directly attacks/incites sexism.
Use JUDGEMENTAL if the text criticizes, condemns, or judges a sexist situation.
"""

    output = "Return only one label: DIRECT or JUDGEMENTAL."

    exemplars = build_fewshot_examples_single_label(train_df, examples_per_label=2)

    if max_examples is not None:
        dev_df = dev_df.head(max_examples).copy()

    predictions = []
    raw_outputs = []

    for _, row in tqdm(dev_df.iterrows(), total=len(dev_df)):
        prompt = create_incontext_few_prompt(
            task_description=task,
            exemplars=exemplars,
            query=row["text"],
            role=role,
            output=output
        )

        response = perform_incontext_classification(
            model,
            tokenizer,
            prompt,
            ntokens=8,
            temp=0.2
        )

        pred = postprocess_task22(response)

        predictions.append(pred)
        raw_outputs.append(response)

    result_df = dev_df.copy()
    result_df["prediction"] = predictions
    result_df["raw_output"] = raw_outputs
    result_df["correct"] = result_df["label"] == result_df["prediction"]

    f1_macro = f1_score(
        result_df["label"],
        result_df["prediction"],
        average="macro"
    )

    print("Subtask 2.2 Few-shot Macro-F1:", f1_macro)
    print(classification_report(result_df["label"], result_df["prediction"], digits=4))

    result_df.to_csv(os.path.join(OUTPUT_DIR, "task22_few_shot_predictions.csv"), index=False)

    return result_df, f1_macro


task22_few_df, task22_few_f1 = run_few_shot_task22(
    model,
    tokenizer,
    train_22,
    dev_22,
    max_examples=50
)

display(task22_few_df.head())

  0%|          | 0/50 [00:00<?, ?it/s]

Subtask 2.2 Few-shot Macro-F1: 0.4318181818181818
              precision    recall  f1-score   support

      DIRECT     0.7600    1.0000    0.8636        38
 JUDGEMENTAL     0.0000    0.0000    0.0000        12

    accuracy                         0.7600        50
   macro avg     0.3800    0.5000    0.4318        50
weighted avg     0.5776    0.7600    0.6564        50



,id,lang,text,label,prediction,raw_output,correct
0,210998,en,"THE CAN'T MEMF @Internet TLCM ""You think there...",DIRECT,DIRECT,DIRECT,True
1,210207,en,ALL MEN ARE PIGS WOMEN ARE EQUAL TO MEN,DIRECT,DIRECT,DIRECT,True
2,211923,en,TRYING TO EXPLAIN WHY 010 WOMEN DON'T HAVE RIG...,DIRECT,DIRECT,DIRECT,True
3,211290,en,> Feminism is just a long con by the patriarch...,DIRECT,DIRECT,DIRECT,True
4,211779,en,"GALS, IF YOU'RE IN AN ARGUMENT WITH A GUY AND ...",DIRECT,DIRECT,DIRECT,True


In [17]:
def build_fewshot_examples_multilabel(train_df, n_examples=6):
    sampled = train_df.sample(
        n=min(n_examples, len(train_df)),
        random_state=SEED
    )

    examples = []

    for _, row in sampled.iterrows():
        examples.append({
            "input": row["text"],
            "output": str(row["label"])
        })

    return examples


def multilabel_f1_macro(y_true, y_pred, labels):
    rows_true = []
    rows_pred = []

    for gold, pred in zip(y_true, y_pred):
        rows_true.append([1 if label in gold else 0 for label in labels])
        rows_pred.append([1 if label in pred else 0 for label in labels])

    return f1_score(rows_true, rows_pred, average="macro", zero_division=0)


def run_few_shot_task23(model, tokenizer, train_df, dev_df, max_examples=None):
    role = "in social psychology and linguistics, specialized in categorizing sexist content"

    task = f"""
Assign one or more sexism categories to the following text.
Possible labels are:
{LABELS_23}

Return a Python-style list with the selected labels.
"""

    output = "Return only a list of labels, for example: ['OBJECTIFICATION', 'SEXUAL-VIOLENCE']."

    exemplars = build_fewshot_examples_multilabel(train_df, n_examples=6)

    if max_examples is not None:
        dev_df = dev_df.head(max_examples).copy()

    predictions = []
    raw_outputs = []

    for _, row in tqdm(dev_df.iterrows(), total=len(dev_df)):
        prompt = create_incontext_few_prompt(
            task_description=task,
            exemplars=exemplars,
            query=row["text"],
            role=role,
            output=output
        )

        response = perform_incontext_classification(
            model,
            tokenizer,
            prompt,
            ntokens=40,
            temp=0.2
        )

        pred = postprocess_task23(response)

        predictions.append(pred)
        raw_outputs.append(response)

    result_df = dev_df.copy()
    result_df["prediction"] = predictions
    result_df["raw_output"] = raw_outputs
    result_df["correct_exact_match"] = [
        sorted(gold) == sorted(pred)
        for gold, pred in zip(result_df["label"], result_df["prediction"])
    ]

    f1_macro = multilabel_f1_macro(
        result_df["label"].tolist(),
        result_df["prediction"].tolist(),
        LABELS_23
    )

    print("Subtask 2.3 Few-shot Macro-F1:", f1_macro)

    result_df.to_csv(os.path.join(OUTPUT_DIR, "task23_few_shot_predictions.csv"), index=False)

    return result_df, f1_macro


task23_few_df, task23_few_f1 = run_few_shot_task23(
    model,
    tokenizer,
    train_23,
    dev_23,
    max_examples=50
)

display(task23_few_df.head())

  0%|          | 0/50 [00:00<?, ?it/s]

Subtask 2.3 Few-shot Macro-F1: 0.4013372095718982


,id,lang,text,label,prediction,raw_output,correct_exact_match
0,210313,en,WE SHOULD HIRE A MAID TO COOK AND CLEAN FOR US...,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, STER...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N...",False
1,211747,en,"LOOK AT ME! LOOK AT ME! LOOK AT ME! ""Tell the ...","[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, SEXU...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N...",False
2,211033,en,facebook.com/greendelabean where is my foond c...,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, STER...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N...",False
3,211926,en,I DON'T ALWAYS KNOW WHAT WOMEN WANT BUT WHEN I...,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N...",False
4,210489,en,Twitter feminists when a man spreads his legs ...,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, SEXU...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N...",False


In [18]:
results_summary = pd.DataFrame([
    {
        "task": "2.1",
        "strategy": "zero-shot",
        "metric": "F1 positive class YES",
        "score": task21_zero_f1,
        "model": MODEL_NAME,
        "n_dev_examples": len(task21_zero_df)
    },
    {
        "task": "2.2",
        "strategy": "few-shot",
        "metric": "Macro-F1",
        "score": task22_few_f1,
        "model": MODEL_NAME,
        "n_dev_examples": len(task22_few_df)
    },
    {
        "task": "2.3",
        "strategy": "few-shot",
        "metric": "Macro-F1 multilabel",
        "score": task23_few_f1,
        "model": MODEL_NAME,
        "n_dev_examples": len(task23_few_df)
    }
])

display(results_summary)

results_summary.to_csv(os.path.join(OUTPUT_DIR, "lab7_results_summary.csv"), index=False)

,task,strategy,metric,score,model,n_dev_examples
0,2.1,zero-shot,F1 positive class YES,0.300000,google/flan-t5-small,50
1,2.2,few-shot,Macro-F1,0.431818,google/flan-t5-small,50
2,2.3,few-shot,Macro-F1 multilabel,0.401337,google/flan-t5-small,50


In [19]:
def show_errors(df, task_name, n=10):
    print(f"Errors for task {task_name}")
    errors = df[df["correct"] == False].head(n) if "correct" in df.columns else df[df["correct_exact_match"] == False].head(n)
    display(errors[["id", "text", "label", "prediction", "raw_output"]])


show_errors(task21_zero_df, "2.1")
show_errors(task22_few_df, "2.2")
show_errors(task23_few_df, "2.3")

Errors for task 2.1


,id,text,label,prediction,raw_output
0,210610,I COMPLETELY SUBSCRIBE TO GENDER ROLES AND FIN...,YES,NO,No
3,210200,THERE MUST BE SOME WAY TO BLAME BLACK MEN... D...,YES,NO,No
4,210262,EVERY MAN WHEN HE SEES NICE BOOBS imgflip.com,YES,NO,No
5,210198,imgflip.com All bitches look alike CHANGE MY M...,YES,NO,No
7,211760,WHY ARE YOU STARING? 133.53,YES,NO,No
9,211956,Aladdin taught me that good things happen to t...,NO,YES,YES
10,210724,me: dysphoria: this is you bm @transboycommuni...,YES,NO,No
11,211216,This is Sally's 3rd promotion this month. I do...,YES,NO,No
16,211082,SMASH OR PASS wwwww ifunny.co,YES,NO,No
17,211969,I HAVE TITS GIVE ME FREE STUFF,YES,NO,No


Errors for task 2.2


,id,text,label,prediction,raw_output
11,211004,"""MAKE ME A SANDWICH"" Torne TAVERSIA.NET I MADE...",JUDGEMENTAL,DIRECT,DIRECT
16,210569,A depiction of nice guys telling women how nic...,JUDGEMENTAL,DIRECT,DIRECT
23,210804,Feminists: I hate all men Imagine if guys said...,JUDGEMENTAL,DIRECT,DIRECT
24,210346,IF WOMEN DO THE SAME JOB AS MEN FOR LESS MONEY...,JUDGEMENTAL,DIRECT,DIRECT
25,211481,How I be looking when she starts acting crazy ...,JUDGEMENTAL,DIRECT,DIRECT
31,211369,O Business Management WHY WOMEN ENTREPRENEURS ...,JUDGEMENTAL,DIRECT,DIRECT
34,210670,Time machine gets invented Girls: SVO I'm your...,JUDGEMENTAL,DIRECT,DIRECT
39,210115,"LST ""women want strong and competent men"" SO y...",JUDGEMENTAL,DIRECT,DIRECT
41,210180,BITCHES BE LIKE.I DON'T DO DRAMA BUT SURE DON'...,JUDGEMENTAL,DIRECT,DIRECT
43,211295,When you're trying to enjoy your day walking d...,JUDGEMENTAL,DIRECT,DIRECT


Errors for task 2.3


,id,text,label,prediction,raw_output
0,210313,WE SHOULD HIRE A MAID TO COOK AND CLEAN FOR US...,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, STER...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
1,211747,"LOOK AT ME! LOOK AT ME! LOOK AT ME! ""Tell the ...","[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, SEXU...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
2,211033,facebook.com/greendelabean where is my foond c...,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, STER...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
3,211926,I DON'T ALWAYS KNOW WHAT WOMEN WANT BUT WHEN I...,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
4,210489,Twitter feminists when a man spreads his legs ...,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, SEXU...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
5,210864,"You know, when you like, you grab a woman's br...","[MISOGYNY-NON-SEXUAL-VIOLENCE, OBJECTIFICATION...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
6,210601,Me: Could you please make dinner? Her: GRAB EM...,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, STER...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
7,210889,MMMMM KILL WOMEN I MUST makeameme.org,"[MISOGYNY-NON-SEXUAL-VIOLENCE, SEXUAL-VIOLENCE...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
8,210182,Are you still taking out batgirl toni- imgflip...,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."
9,211863,"imgflip.com YOUR DAUGHTER DRESSES LIKE A HO, A...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...","Output: ['IDEOLOGICAL-INEQUALITY', 'MISOGYNY-N..."



# 1. Introduction

The objective of this lab session was to apply Large Language Models (LLMs) to the EXIST2026 sexism detection tasks using prompting and in-context learning techniques.

Unlike previous labs, in this session the model is not fine-tuned. Instead, the approach consists of using a pre-trained instruction-following language model and guiding its behaviour through prompts.

The tasks covered in this practice are:

- **Subtask 2.1:** Sexism identification (`YES` / `NO`)
- **Subtask 2.2:** Source intention classification (`DIRECT` / `JUDGEMENTAL`)
- **Subtask 2.3:** Multi-label sexism categorisation

The main objective was not only obtaining predictions, but also understanding how prompt engineering, few-shot examples and postprocessing affect model behaviour.

---

# 2. Environment and execution issues

The notebook was executed in Google Colab.

Originally, the reference notebook proposed the use of larger LLMs, including LLaMA-based architectures. However, several practical limitations appeared during execution:

- GPU quota became unavailable during the session.
- Some libraries required by the notebook generated dependency conflicts.
- The environment suffered compatibility issues between `numpy`, `scipy` and `ferret-xai`.
- Some installations froze while compiling packages without GPU acceleration.
- HuggingFace authentication warnings also appeared because no `HF_TOKEN` was configured.

Because of these limitations, the notebook had to be adapted in order to work correctly using CPU execution.

Several package reinstallations were necessary to stabilize the environment. In particular:

- incompatible `numpy` and `scipy` versions had to be downgraded,
- missing dependencies such as `thermostat` had to be manually installed,
- and some cells from the original notebook had to be skipped or modified.

During the process, an accidental deletion of the mounted Google Drive folder also occurred after executing a shell command over `/content/drive`. Fortunately, the folder was recovered from the Google Drive trash without data loss.

After recovering the environment and reinstalling the required dependencies, the notebook was successfully executed.

---

# 3. Model selection

Due to the lack of GPU resources and the memory limitations of Colab CPU execution, a lightweight instruction-tuned model was selected:

```python
google/flan-t5-small
```

This model is significantly smaller than LLaMA-based alternatives, but it supports instruction following and text generation tasks.

The choice of this model allowed the notebook to execute completely under CPU constraints.

The main trade-off is that the model has lower reasoning and classification capacity compared to larger instruction-tuned models. This directly affects final performance metrics.

---

# 4. Dataset preprocessing

The dataset used in this lab was:

```python
EXIST2026_training.json
```

The original dataset contains multiple annotations per example. Therefore, preprocessing was necessary before evaluation.

For subtask 2.1, which is a binary classification problem, majority voting was applied over all annotator labels in order to obtain a single final label for each example. The possible labels were:

- `YES`
- `NO`

Examples without a clear majority were discarded.

For subtask 2.2, only examples classified as sexist in task 2.1 were considered. Again, majority voting was applied to obtain a single label between:

- `DIRECT`
- `JUDGEMENTAL`

This task is more difficult because it requires understanding whether the text promotes sexism or criticizes sexist behaviour.

For subtask 2.3, a multi-label setup was required. The following labels were used:

- `IDEOLOGICAL-INEQUALITY`
- `STEREOTYPING-DOMINANCE`
- `MISOGYNY-NON-SEXUAL-VIOLENCE`
- `OBJECTIFICATION`
- `SEXUAL-VIOLENCE`

Since examples may belong to multiple categories simultaneously, the final labels were created as the union of positive labels assigned by annotators.

After preprocessing, English examples were separated and train/dev splits were generated for evaluation.

---

# 5. Prompt engineering

The notebook implements both zero-shot and few-shot prompting approaches.

For subtask 2.1, zero-shot prompting was used. The prompt included:

- an expert role,
- a task description,
- output constraints,
- and the target text.

The model received no labelled examples.

The objective was to evaluate whether the model could detect sexist content using only instructions and prior knowledge.

For subtasks 2.2 and 2.3, few-shot prompting was implemented.

In this approach, labelled examples were inserted inside the prompt before the target example.

Each demonstration followed the format:

```text
Input: ...
Output: ...
```

The few-shot strategy helps the model infer the expected classification behaviour from examples.

This was especially useful for:
- source intention classification,
- and multi-label categorisation.

These tasks require more contextual and semantic understanding than simple binary classification.

---

# 6. Output postprocessing

A major issue when using generative models for classification is that the model does not directly output class IDs.

Instead, it generates free natural language text.

For example, instead of generating only:

```text
YES
```

the model may produce:

```text
Yes, this sentence contains sexist content.
```

Therefore, postprocessing functions were implemented in order to map generated outputs back into valid labels.

For subtask 2.1, outputs were mapped into:
- `YES`
- `NO`

For subtask 2.2, outputs were mapped into:
- `DIRECT`
- `JUDGEMENTAL`

For subtask 2.3, generated text was searched for valid category names.

This postprocessing stage was essential for evaluation because the model frequently generated explanations or additional text.

---

# 7. Results

The following results were obtained using `google/flan-t5-small` executed on CPU.

| Task | Strategy | Metric | Score |
|---|---|---|---:|
| 2.1 | Zero-shot | F1 positive class YES | 0.3000 |
| 2.2 | Few-shot | Macro-F1 | 0.4318 |
| 2.3 | Few-shot | Macro-F1 multilabel | 0.4013 |

---

# 8. Analysis of results

For subtask 2.1, the zero-shot setup achieved an F1 score of approximately `0.30`.

The model tended to overpredict the `NO` class. This behaviour is visible in the confusion matrix and error examples.

The result suggests that small instruction-tuned models struggle to identify implicit sexism without labelled demonstrations.

However, the model still correctly classified many explicit non-sexist examples.

For subtask 2.2, few-shot prompting improved performance and achieved a Macro-F1 score of approximately `0.43`.

The model correctly identified most `DIRECT` examples, but frequently failed on `JUDGEMENTAL` samples.

The main issue observed was class bias. The model strongly preferred the `DIRECT` label even when the text was criticizing sexism rather than expressing it.

This indicates that source intention classification requires deeper pragmatic reasoning.

For subtask 2.3, the Macro-F1 score was approximately `0.40`.

This was the most difficult task because:
- it is multi-label,
- labels overlap semantically,
- and the model must assign several categories simultaneously.

Several problems appeared:
- the model tended to repeat the same categories,
- predictions often included incomplete label sets,
- and generated outputs sometimes contained explanations instead of only labels.

Despite these limitations, the model was still capable of generating partially correct predictions.

---

# 9. Error analysis

The error analysis confirmed several limitations of prompt-based classification using small LLMs.

The most common errors observed were:

- confusion between mentioning sexism and expressing sexism,
- failure to detect implicit sexist meaning,
- overprediction of majority classes,
- incorrect multi-label assignments,
- and noisy generated outputs.

In subtask 2.2, many `JUDGEMENTAL` examples were incorrectly predicted as `DIRECT`.

In subtask 2.3, the model frequently predicted:

```text
MISOGYNY-NON-SEXUAL-VIOLENCE
```

for many different examples, suggesting a strong label bias.

This behaviour is likely caused by:
- limited model size,
- CPU-only execution constraints,
- reduced contextual reasoning capacity,
- and the complexity of the task itself.

---

# 10. Main difficulties encountered

Several technical problems appeared during implementation:

- dependency conflicts between libraries,
- incompatible `numpy` and `scipy` versions,
- installation freezes during package compilation,
- missing dependencies such as `thermostat`,
- CPU-only execution,
- long inference times,
- and HuggingFace authentication warnings.

Because GPU acceleration was unavailable, execution speed became significantly slower.

Some inference cells required several minutes to complete.

This forced the use of:
- a smaller instruction model,
- reduced evaluation subsets,
- and lighter configurations.

During execution, an accidental deletion of the mounted Google Drive folder also occurred due to a shell command executed over `/content/drive`. Fortunately, the folder was recovered from the Google Drive trash and no data was permanently lost.

---

# 11. Conclusions

This lab demonstrated how Large Language Models can be applied to classification tasks using prompting and in-context learning instead of fine-tuning.

The experiments showed that prompt engineering requires:
- carefully designed prompts,
- representative examples,
- robust postprocessing,
- and proper evaluation procedures.

The results confirm that:
- zero-shot prompting works better for simpler binary tasks,
- few-shot prompting improves performance on more complex tasks,
- and multi-label classification remains challenging for small instruction-tuned models.

Although the obtained scores are modest, they are coherent with:
- the selected lightweight model,
- CPU-only execution,
- and the complexity of the EXIST2026 tasks.

Overall, the practice successfully implemented the complete prompting workflow:
- dataset preprocessing,
- prompt construction,
- zero-shot inference,
- few-shot inference,
- postprocessing,
- evaluation,
- and error analysis.

Possible future improvements include:
- using larger instruction-tuned models,
- enabling GPU acceleration,
- improving prompt templates,
- retrieval-based example selection,
- and more advanced decoding strategies.